# Daltonize Algorithm: TensorFlow Implementation

This notebook implements color blindness simulation and correction using TensorFlow 2.

The algorithm transforms images through sRGB → LMS → Simulation/Correction → LMS → sRGB color spaces.

In [ ]:
import tensorflow as tf
import numpy as np

# Transformation matrices for color space conversions and simulations
# Based on the Daltonize algorithm (https://github.com/joergdietrich/daltonize)

# sRGB to LMS conversion matrix
RGB_TO_LMS = tf.constant([
    [0.3, 0.622, 0.078],
    [0.23, 0.692, 0.078],
    [0.25, 0.125, 0.625]
], dtype=tf.float32)

# LMS to sRGB conversion matrix (inverse of RGB_TO_LMS)
LMS_TO_RGB = tf.constant([
    [11.031, -9.38, -0.651],
    [-3.254, 2.414, -0.16],
    [-3.66, 3.283, 0.377]
], dtype=tf.float32)

# Protanopia (Red blindness) simulation matrix
PROT_SIM_MATRIX = tf.constant([
    [0.0, 2.02344, -2.52581],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
], dtype=tf.float32)

# Deuteranopia (Green blindness) simulation matrix
DEUT_SIM_MATRIX = tf.constant([
    [1.0, 0.0, 0.0],
    [0.494207, 0.0, 1.24827],
    [0.0, 0.0, 1.0]
], dtype=tf.float32)

# Tritanopia (Blue blindness) simulation matrix
TRIT_SIM_MATRIX = tf.constant([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [-0.395913, 0.801109, 0.0]
], dtype=tf.float32)

# Correction matrices (inverse of simulation matrices)
PROT_CORRECTION_MATRIX = tf.constant([
    [0.0, -0.4942, 1.1945],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
], dtype=tf.float32)

DEUT_CORRECTION_MATRIX = tf.constant([
    [1.0, 0.0, 0.0],
    [-0.882516, 0.0, -0.805306],
    [0.0, 0.0, 1.0]
], dtype=tf.float32)

TRIT_CORRECTION_MATRIX = tf.constant([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.494207, -1.000000, 0.0]
], dtype=tf.float32)

## DaltonizeLayer: Keras Custom Layer

Implements color blindness simulation and correction for images.
- **Input shape:** (Batch, Height, Width, 3)
- **Output shape:** (Batch, Height, Width, 3)

In [ ]:
class DaltonizeLayer(tf.keras.layers.Layer):
    """
    Custom Keras layer for simulating or correcting color blindness.
    
    Args:
        deficiency_type (str): Type of color blindness ('protanopia', 'deuteranopia', 'tritanopia')
        correction (bool): If True, apply correction; if False, apply simulation (default=False)
    """
    
    def __init__(self, deficiency_type='deuteranopia', correction=False, **kwargs):
        super(DaltonizeLayer, self).__init__(**kwargs)
        self.deficiency_type = deficiency_type.lower()
        self.correction = correction
        
        # Select the appropriate matrices
        if self.deficiency_type == 'protanopia':
            self.sim_matrix = PROT_SIM_MATRIX
            self.corr_matrix = PROT_CORRECTION_MATRIX
        elif self.deficiency_type == 'deuteranopia':
            self.sim_matrix = DEUT_SIM_MATRIX
            self.corr_matrix = DEUT_CORRECTION_MATRIX
        elif self.deficiency_type == 'tritanopia':
            self.sim_matrix = TRIT_SIM_MATRIX
            self.corr_matrix = TRIT_CORRECTION_MATRIX
        else:
            raise ValueError(f"Unknown deficiency type: {deficiency_type}")
        
        # Select the transformation matrix based on mode
        self.transform_matrix = self.corr_matrix if self.correction else self.sim_matrix
    
    def call(self, inputs):
        """
        Apply Daltonize transformation to input images.
        
        Args:
            inputs: Tensor of shape (Batch, Height, Width, 3) with values in [0, 1]
        
        Returns:
            Transformed image tensor of the same shape
        """
        # Store original shape and flatten spatial dimensions
        original_shape = tf.shape(inputs)
        batch_size = original_shape[0]
        height = original_shape[1]
        width = original_shape[2]
        
        # Reshape to (Batch * Height * Width, 3) for matrix multiplication
        flat_images = tf.reshape(inputs, [-1, 3])
        
        # Step 1: Convert sRGB to LMS
        lms = tf.matmul(flat_images, tf.transpose(RGB_TO_LMS))
        
        # Step 2: Apply simulation or correction matrix
        transformed_lms = tf.matmul(lms, tf.transpose(self.transform_matrix))
        
        # Step 3: Convert back from LMS to sRGB
        output = tf.matmul(transformed_lms, tf.transpose(LMS_TO_RGB))
        
        # Reshape back to original image dimensions
        output = tf.reshape(output, original_shape)
        
        # Clamp values to [0, 1] to maintain valid image range
        output = tf.clip_by_value(output, 0.0, 1.0)
        
        return output
    
    def get_config(self):
        """Return config for serialization."""
        config = super().get_config()
        config.update({
            'deficiency_type': self.deficiency_type,
            'correction': self.correction
        })
        return config